In [ ]:
import numpy as np
import pandas as pd
from my_helpers import *

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Below identifiers correspond to evaluation protocol 1, 2 and 3 respectively
# identifiers - "TouchPose_crossSession", "TouchPose_crossSubjects", "TouchPose_crossGestures"
identifier = "TouchPose_crossGestures"
data_path = "../data/eth_data/"
model_path = "../models/"
new_data = "../data/27Arish"

# Information from Report

The valid skeleton data we collected from leap motion controller covered 147 trails,
with an average number of frames per trail of 1316. The frame rate of Leap motion data is
30 fps.

The mediapipe skeleton data we collected from hand writing on Wacom tablet covered
200 trails. The average number of frames per trial is 1540. The working fps of RGB camera
is 30 fps.

The capacity image and stylus data were recorded from Wacom tablet. The capacitive
image has the resolution 72*42. We collected 132 trails of capacity image and 201 trails
of pen data in total. The average number of frames of capacitive data per trial is 13203
frames, and the number of frames of pen data per trial is 12262. The recording rate is
about 80 fps.

In [ ]:
cap_img = read_cap_img(new_data+"/Arish_1_UP_100_rawdata.csv")
skeleton = read_skeleton(new_data+"/userarish_2023-04-03_10-10-07_11956_handleap.csv")

In [ ]:
skeleton.Timestamp = skeleton.Timestamp*1000
skeleton.Timestamp = skeleton.Timestamp.astype('int64')
cap_img = cap_img.rename(columns={'TimeStamps': 'Timestamp'})
cap_img.head()

In [ ]:
skeleton.Timestamp.diff().mean()

In [ ]:
cap_img.Timestamp.diff().mean()

It looks like MergeTimeStamps is the biggest smaller value mutiple of 5 with respect to TimeStamps.
We do the same for skeleton.

In [ ]:
# we will make both df overlap
print(skeleton.iloc[0].Timestamp < cap_img.iloc[0].Timestamp)
print(skeleton.iloc[-1].Timestamp > cap_img.iloc[-1].Timestamp)

Get overlapping between skeleton and capactive images

In [ ]:
mask = (skeleton.Timestamp >= cap_img.Timestamp.iloc[0]) & (skeleton.Timestamp <= cap_img.Timestamp.iloc[-1])
mask[mask==True]

Hence we will also take rows 760 and 4254

In [ ]:
loc1=mask[mask==True].first_valid_index()
locn=mask[mask==True].last_valid_index()
mask.loc[loc1-1]=True
mask.loc[locn+1]=True

In [ ]:
mask[mask==True]

In [ ]:
skeleton=skeleton[mask].reset_index(drop=True)
len(skeleton)

In [ ]:
len(cap_img.img)

Hence we want to get 3496 observations, we will map skeleton to average of capactive images.

There are some big jumps in skeleton data

In [ ]:
skeleton.Timestamp.diff().sort_values(ascending=False)  

In [ ]:
cap_img.Timestamp.diff().sort_values(ascending=False)  

In [ ]:
bins = skeleton.Timestamp
x = cap_img.Timestamp
cap_img['where_to'] = np.digitize(x, bins)
cap_img.where_to=cap_img.where_to.map(lambda x: x-1)
cap_img.where_to.unique()

In [ ]:
ranking =skeleton.Timestamp.diff().sort_values(ascending=False)
ranking=ranking[ranking > 80]
ranking.index=ranking.index-1
ranking

In [ ]:
cap_img.where_to.value_counts()

we remove these

In [ ]:
prev = len(cap_img)
cap_img=cap_img[~cap_img.where_to.isin(ranking.index)]
print('discarded frames', prev - len(cap_img))
res=cap_img.groupby('where_to')['img'].agg(lambda x : np.mean(np.array(x)))

In [ ]:
data = skeleton.copy()
data['img'] = res
data=data.dropna().reset_index(drop=True)
print('discarded frames', len(skeleton) - len(data))

In [ ]:
hand_cords = data.drop(["img","Timestamp"],axis=1)
data['skeleton'] = hand_cords.values.tolist()
data = data.drop(hand_cords.columns, axis=1)
data.skeleton = data.skeleton.map(lambda x: np.array(x).reshape(21,3))
data.to_pickle('../data/gen_data/Arish_1_UP_100_.pkl')

In [ ]:
data